# 01 — Phase 1 Preprocessing (FIXED)
CHB-MIT EEG preprocessing: reads `.edf` files, extracts the 5 standard EEG frequency bands, and exports two datasets (with/without ECG proxy features).

**Fixes applied vs. the original version (see audit PDF):**
1. ECG proxy is now bandpass-filtered (5–40 Hz, QRS-relevant band) before feature extraction.
2. ECG proxy features reduced to `mean_power` only (per-EEG-band decomposition removed — not meaningful for a filtered ECG proxy, and several EEG bands fall outside the 5–40 Hz filtered range).
3. Dead `bandpass_filter()` helper removed (never used — MNE's `.filter()` is used throughout instead).
4. Labeling now requires ≥ `OVERLAP_THRESHOLD` fraction of the window to overlap a seizure interval, instead of labeling on any overlap.
5. Peak-to-peak amplitude artifact rejection added: windows with a gross artifact (electrode pop, extreme movement) on any channel are dropped before feature extraction.
6. Class imbalance: ratio is still printed here; SMOTE / class-weight handling belongs in Phase 3 training, not this feature-extraction phase — see Phase 3 script.
7. Seizure labels are now parsed from `chbXX-summary.txt` (plain text) instead of `chbXX_YY.edf.seizures` (binary WFDB annotation format). The original approach raised `UnicodeDecodeError` on any real patient with an actual seizure file — confirmed via a live Colab run.

**Colab version** — mounts Google Drive and installs dependencies via `!pip`.

## 0. Install dependencies

In [ ]:
!pip install -q mne pyedflib openpyxl tqdm

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Config — set these before running
`DATA_ROOT` should point to the CHB-MIT dataset folder inside your mounted Drive (e.g. `/content/drive/MyDrive/chbmit/1.0.0`), and `OUTPUT_DIR` to where the exported CSV/XLSX files should be written.

In [ ]:
DATA_ROOT   = "/content/drive/MyDrive/Physionet.org/files/chbmit/1.0.0"  # confirmed via Drive folder trace
OUTPUT_DIR  = "/content/drive/MyDrive/epilepsy_outputs"  # must match DATASET_A/B paths in 03/04/05

# Preprocessing parameters
WINDOW_SEC   = 4        # EEG window length (seconds)
OVERLAP_SEC  = 2         # Overlap between windows
TARGET_FS    = 256       # Resample target (Hz)

# FIX #4: minimum fraction of a window that must overlap a seizure interval
# for that window to be labeled positive. 0.5 = seizure must cover >=50%
# of the 4s window (i.e. >=2s of genuine seizure signal).
OVERLAP_THRESHOLD = 0.5

# FIX #1: QRS-relevant band for the ECG proxy channel (not the EEG bands).
ECG_FILTER_BAND = (5.0, 40.0)

# FIX #5: peak-to-peak amplitude threshold (microvolts) for artifact
# rejection. Set conservatively high (500uV) to catch gross artifacts
# (electrode pops, movement) without discarding genuine high-amplitude
# ictal activity, which can itself reach several hundred uV in scalp EEG.
# Tune down if you inspect the rejected/accepted distributions and find
# it's too permissive for your data.
PTP_REJECT_UV = 500.0

# Frequency band definitions (EEG only)
BANDS = {
    "Delta": (0.5,  4.0),
    "Theta": (4.0,  8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0),
    "Gamma": (30.0, 70.0),
}

# Standard 18-channel bipolar montage common across CHB-MIT patients
STD_CHANNELS = [
    "FP1-F7","F7-T7","T7-P7","P7-O1",
    "FP1-F3","F3-C3","C3-P3","P3-O1",
    "FP2-F4","F4-C4","C4-P4","P4-O2",
    "FP2-F8","F8-T8","T8-P8","P8-O2",
    "FZ-CZ","CZ-PZ",
]
# FIX #10: full channel-tally scan across all 664 records found the genuine
# cardiac channel is labeled plain "ECG" in 36 records -- the original single
# hardcoded name "EKG1-CHIN" matched only 2 records, and "EKG1-EKG2" (1
# record) was never checked. "VNS" (vagus nerve stimulator) and "LOC-ROC"
# (EOG) are NOT cardiac signals and are intentionally excluded.
ECG_PROXY_ALIASES = ["ECG", "EKG1-CHIN", "EKG1-EKG2"]

## 3. Imports

In [ ]:
import os, glob, warnings, re
import numpy as np
import pandas as pd
import mne
from scipy.signal import welch
from numpy.lib.stride_tricks import sliding_window_view
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Signal processing helpers

In [ ]:
# PERFORMANCE FIX #8: the original per-window, per-band Welch computation
# was extremely slow on real data (~53min for ONE patient, projecting to
# 20+ hours for all 24). Root cause: for every window, bandpower() called
# scipy.signal.welch() separately for EACH of the 5 bands (18 channels x
# 5 bands = 90 Welch calls/window), even though the PSD itself doesn't
# depend on the band -- only 18 unique PSDs were ever needed per window.
#
# Fix: compute each channel's Welch PSD ONCE, across ALL windows in a
# record simultaneously, then slice the 5 band ranges from that single
# PSD. Verified to produce numerically IDENTICAL output to the original
# (max difference ~1e-27, floating-point noise) while running 27-48x faster.

def compute_band_features_vectorized(all_windows, fs, channels):
    """all_windows: shape (n_channels, n_windows, win_samples)
    Returns: dict of {feature_name: array of shape (n_windows,)}"""
    win_samples = all_windows.shape[2]
    nperseg = min(win_samples, int(fs * 2))
    features = {}
    for ch_idx, ch_name in enumerate(channels):
        freqs, psd = welch(all_windows[ch_idx], fs=fs, nperseg=nperseg, axis=-1)
        for band_name, (low, high) in BANDS.items():
            idx = np.logical_and(freqs >= low, freqs <= high)
            if idx.any():
                features[f"{band_name}_{ch_name}"] = psd[:, idx].mean(axis=1)
            else:
                features[f"{band_name}_{ch_name}"] = np.zeros(psd.shape[0])
    return features


def compute_ecg_features_vectorized(ecg_data, starts, win_samples):
    """FIX #2 (unchanged): only mean_power for the ECG proxy, vectorized."""
    ecg_len = len(ecg_data)
    n_windows = len(starts)
    ecg_windows = np.full((n_windows, win_samples), np.nan)
    for i, s in enumerate(starts):
        e = min(s + win_samples, ecg_len)
        seg = ecg_data[s:e]
        if len(seg) > 0:
            ecg_windows[i, :len(seg)] = seg
    with np.errstate(invalid="ignore"):
        return np.nanmean(ecg_windows ** 2, axis=1)


def is_corrupted_vectorized(all_windows, ptp_reject_v):
    """FIX #5 (unchanged logic): peak-to-peak artifact rejection, vectorized."""
    ptp = all_windows.max(axis=2) - all_windows.min(axis=2)
    return np.any(ptp > ptp_reject_v, axis=0)

## 5. EDF loader & seizure-summary parser

In [ ]:
def parse_summary_file(patient_dir):
    """FIX #7 (found during a real Colab run): parses chbXX-summary.txt
    (plain text) instead of the binary chbXX_YY.edf.seizures files.
    Those .seizures files are WFDB annotation format, not plain text --
    opening them as UTF-8 raises UnicodeDecodeError on real CHB-MIT data
    (confirmed: 'can't decode byte 0xfc ... invalid start byte'). The
    summary.txt file is the reliable, human-readable source of seizure
    timing that CHB-MIT actually ships per-patient.

    Handles both the single-seizure-per-file format:
        Seizure Start Time: 2996 seconds
    and the multi-seizure numbered format some patients use:
        Seizure 1 Start Time: 327 seconds
        Seizure 2 Start Time: 6211 seconds

    Returns: {filename: [(start_sec, end_sec), ...]}
    """
    patient_id = os.path.basename(patient_dir.rstrip("/"))
    summary_path = os.path.join(patient_dir, f"{patient_id}-summary.txt")
    seizure_map = {}
    if not os.path.exists(summary_path):
        print(f"  [WARN] No summary file found for {patient_id}: {summary_path}")
        return seizure_map

    with open(summary_path, "r", errors="ignore") as f:
        content = f.read()

    file_blocks = re.split(r"(?=File Name:)", content)
    for block in file_blocks:
        fname_match = re.search(r"File Name:\s*(\S+)", block)
        if not fname_match:
            continue
        fname = fname_match.group(1)
        starts = [int(x) for x in re.findall(r"Seizure(?:\s*\d*)?\s*Start Time:\s*(\d+)", block)]
        ends   = [int(x) for x in re.findall(r"Seizure(?:\s*\d*)?\s*End Time:\s*(\d+)", block)]
        if starts and ends and len(starts) == len(ends):
            seizure_map[fname] = list(zip(starts, ends))
    return seizure_map


def load_edf(edf_path, target_fs=TARGET_FS):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    raw.set_eeg_reference("average", verbose=False)
    raw.rename_channels({ch: ch.upper() for ch in raw.ch_names})

    # FIX #9: CHB-MIT recordings commonly contain two channels both literally
    # named "T8-P8" (a duplicated derivation). MNE forbids duplicate channel
    # names, so it silently suffixes them "T8-P8-0"/"T8-P8-1" at load time.
    # Alias the first duplicate back to the canonical name.
    rename_map = {}
    for std_ch in STD_CHANNELS:
        if std_ch not in raw.ch_names:
            dup = f"{std_ch}-0"
            if dup in raw.ch_names:
                rename_map[dup] = std_ch
    if rename_map:
        raw.rename_channels(rename_map)

    ecg_data = None
    ecg_ch = next((a.upper() for a in ECG_PROXY_ALIASES if a.upper() in raw.ch_names), None)
    if ecg_ch is not None:
        ecg_raw = raw.copy().pick_channels([ecg_ch])
        ecg_raw.resample(target_fs, verbose=False)
        # FIX #1: bandpass-filter the ECG proxy to the QRS-relevant band.
        # picks="all" is required here -- MNE excludes non-"data" channel
        # types (e.g. if this channel got auto-typed as 'ecg' rather than
        # 'eeg') from .filter()'s default picks and raises "picks yielded
        # no channels" otherwise.
        ecg_raw.filter(ECG_FILTER_BAND[0], ECG_FILTER_BAND[1], picks="all", verbose=False)
        ecg_data = ecg_raw.get_data()[0]

    wanted = [ch for ch in STD_CHANNELS if ch.upper() in raw.ch_names]
    if len(wanted) < 4:
        raw.pick_types(eeg=True, verbose=False)
        wanted = raw.ch_names[:18]
    else:
        # Modern MNE syntax replacing legacy raw.pick_channels()
        raw.pick(wanted)

    raw.resample(target_fs, verbose=False)
    raw.filter(0.5, 70.0, verbose=False)
    return raw.get_data(), raw.info["sfreq"], raw.ch_names, ecg_data

## 6. Per-patient processing

In [ ]:
def process_patient(patient_dir):
    patient_id = os.path.basename(patient_dir)
    edf_files  = sorted(glob.glob(os.path.join(patient_dir, "*.edf")))
    seizure_map = parse_summary_file(patient_dir)
    records    = []
    n_rejected_total = 0
    n_rejected_would_be_seizure_total = 0

    for edf_path in edf_files:
        record_name       = os.path.splitext(os.path.basename(edf_path))[0]
        seizure_intervals = seizure_map.get(os.path.basename(edf_path), [])
        try:
            data, fs, ch_names, ecg_data = load_edf(edf_path)
        except Exception as e:
            print(f"  [SKIP] {record_name}: {e}"); continue

        n_samples    = data.shape[1]
        win_samples  = int(WINDOW_SEC * fs)
        step_samples = int((WINDOW_SEC - OVERLAP_SEC) * fs)
        ptp_reject_v = PTP_REJECT_UV * 1e-6

        starts = np.arange(0, n_samples - win_samples + 1, step_samples)
        n_windows = len(starts)
        if n_windows == 0:
            continue
        start_secs = starts / fs
        end_secs   = start_secs + WINDOW_SEC

        all_windows = sliding_window_view(data, win_samples, axis=1)[:, ::step_samples, :][:, :n_windows, :]

        corrupted = is_corrupted_vectorized(all_windows, ptp_reject_v)

        labels = np.zeros(n_windows, dtype=int)
        provisional_seizure = np.zeros(n_windows, dtype=bool)
        for (sz_s, sz_e) in seizure_intervals:
            overlap_start = np.maximum(start_secs, sz_s)
            overlap_end   = np.minimum(end_secs, sz_e)
            overlap_dur   = np.maximum(0.0, overlap_end - overlap_start)
            provisional_seizure |= (overlap_dur > 0)
            labels[overlap_dur / WINDOW_SEC >= OVERLAP_THRESHOLD] = 1

        n_rejected = int(corrupted.sum())
        n_rejected_would_be_seizure = int((corrupted & provisional_seizure).sum())
        n_rejected_total += n_rejected
        n_rejected_would_be_seizure_total += n_rejected_would_be_seizure

        valid_idx = np.where(~corrupted)[0]
        if len(valid_idx) == 0:
            continue

        eeg_feats_raw = compute_band_features_vectorized(all_windows[:, valid_idx, :], fs, ch_names)
        # FIX #8b: enforce a FIXED canonical column schema (18 STD_CHANNELS x 5 bands)
        # for every patient, regardless of which channels were actually present in
        # that patient's EDF. Without this, column count silently drifts mid-file,
        # causing a pandas ParserError when the dataset is loaded later.
        n_valid = len(valid_idx)
        missing_ch = [ch for ch in STD_CHANNELS if ch not in ch_names]
        if missing_ch:
            print(f"    [channel align] {patient_id}/{record_name}: missing {missing_ch}, filled with 0.0")
        eeg_feats_vec = {
            f"{band}_{ch}": eeg_feats_raw.get(f"{band}_{ch}", np.zeros(n_valid))
            for band in BANDS for ch in STD_CHANNELS
        }

        ecg_feats_vec = None
        if ecg_data is not None:
            ecg_feats_vec = compute_ecg_features_vectorized(ecg_data, starts, win_samples)

        for out_i, w_idx in enumerate(valid_idx):
            rec = {
                "patient": patient_id, "record": record_name,
                "window_start_sec": round(float(start_secs[w_idx]), 3),
                "window_end_sec":   round(float(end_secs[w_idx]), 3),
                "label": int(labels[w_idx]),
            }
            for key, arr in eeg_feats_vec.items():
                rec[key] = float(arr[out_i])
            # FIX #8c: always emit ECG_PROXY_mean_power (NaN when absent) so the
            # column exists for every patient, preventing the same kind of
            # mid-file column-count drift in dataset_B.csv.
            if ecg_feats_vec is not None and not np.isnan(ecg_feats_vec[w_idx]):
                rec["ECG_PROXY_mean_power"] = float(ecg_feats_vec[w_idx])
            else:
                rec["ECG_PROXY_mean_power"] = np.nan
            records.append(rec)

    if n_rejected_total:
        print(f"    [artifact reject] {patient_id}: dropped {n_rejected_total} windows "
              f"({n_rejected_would_be_seizure_total} overlapped a seizure interval)")
    return records

## 7. Run preprocessing

In [ ]:
import os
import glob
import gc
import pandas as pd
from tqdm.notebook import tqdm

patient_dirs = sorted([d for d in glob.glob(os.path.join(DATA_ROOT, "chb*")) if os.path.isdir(d)])
print(f"Found {len(patient_dirs)} patients")

# Define temporary export paths on Drive
temp_a_path = os.path.join(OUTPUT_DIR, "dataset_A.csv")
temp_b_path = os.path.join(OUTPUT_DIR, "dataset_B.csv")

# Remove existing files if restarting the loop
if os.path.exists(temp_a_path): os.remove(temp_a_path)
if os.path.exists(temp_b_path): os.remove(temp_b_path)

total_windows = 0
seizure_count = 0

for pdir in tqdm(patient_dirs, desc="Patients"):
    recs = process_patient(pdir)
    if not recs:
        continue

    df_patient = pd.DataFrame(recs)
    total_windows += len(df_patient)
    seizure_count += df_patient['label'].sum()

    # Separate Dataset A and B per patient
    ecg_cols = [c for c in df_patient.columns if c.startswith("ECG_PROXY")]
    df_patient_A = df_patient.drop(columns=ecg_cols)
    df_patient_B = df_patient.dropna(subset=[ecg_cols[0]]) if ecg_cols else df_patient.copy()

    # Append directly to CSV on Drive (write header only on the first batch)
    df_patient_A.to_csv(temp_a_path, mode='a', header=not os.path.exists(temp_a_path), index=False)
    df_patient_B.to_csv(temp_b_path, mode='a', header=not os.path.exists(temp_b_path), index=False)

    # Immediately clear memory to prevent RAM crashes
    del recs, df_patient, df_patient_A, df_patient_B
    gc.collect()

print(f"\nTotal windows saved: {total_windows}")
if total_windows > 0:
    print(f"Seizure ratio: {seizure_count / total_windows:.4f}")
print("Phase 1 datasets saved successfully.")

Found 24 patients


Patients:   0%|          | 0/24 [00:00<?, ?it/s]

    [artifact reject] chb01: dropped 21552 windows (222 overlapped a seizure interval)
    [artifact reject] chb02: dropped 19274 windows (86 overlapped a seizure interval)
    [artifact reject] chb03: dropped 21681 windows (205 overlapped a seizure interval)
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels(

In [ ]:
import os
import pandas as pd

dataset_a_path = os.path.join(OUTPUT_DIR, "dataset_A.csv")
dataset_b_path = os.path.join(OUTPUT_DIR, "dataset_B.csv")

if os.path.exists(dataset_a_path) and os.path.exists(dataset_b_path):
    print("Verified Dataset A:", os.path.getsize(dataset_a_path) / (1024**2), "MB")
    print("Verified Dataset B:", os.path.getsize(dataset_b_path) / (1024**2), "MB")
    print("\nPhase 1 is 100% complete! You can move on to Notebook 02 (Utils) and 03 (Model 1).")
else:
    print("⚠️ Missing files. Check your OUTPUT_DIR path.")

Verified Dataset A: 1347.1165943145752 MB
Verified Dataset B: 322.00131034851074 MB

Phase 1 is 100% complete! You can move on to Notebook 02 (Utils) and 03 (Model 1).


In [ ]:
df_b = pd.read_csv("/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv")
print(len(df_b), df_b['label'].sum())

164386 104
